# Connectome figure panels

Three headline panels, one per claim (CLAUDE.md, "Scientific objective"),
plus three domain-restricted `domain_*.png` panels — a robustness-tier check
on claim 2 (movies, videogames, stories; see CLAUDE.md, "Domain-restricted
robustness figures") placed in the same montage at the user's request.
Reads `output_data/group_stats/*.tsv` (written by `run-group-stats`) and
plots only — no connectome loading, no similarity computation, which now
lives in `analysis/group_stats.py`.

1. `longitudinal.png` — claim 1 (stable across five years): within-subject
   Pearson similarity vs. friends season lag, one line per network, against
   the between-subject floor.
2. `cross_context.png` — claim 2 (captures a variety of functional brain
   states): the four same/different-subject x same/different-dataset bins,
   all datasets, all networks.
3. `network_quality.png` — claim 3 (applies to all networks, with varying
   quality): per-network within-subject stability vs. median tSNR (falls back
   to a labelled ordering plot with a coverage note when tSNR coverage is too
   thin — see CLAUDE.md, "The QC measures asset").
4. `domain_movies.png` / `domain_videogames.png` / `domain_stories.png` — the
   same four-bin comparison as `cross_context.png`, restricted one domain at a
   time to `analysis.group_stats.DOMAIN_DATASETS`. Movies uses title-level task
   identity (friends season or movie10 title); videogames/stories reuse
   dataset-level identity, same as `cross_context.png`.

**Panels carry no legend and no title.** Both are montage-level furniture, and
inside a panel-sized canvas they crowd the data — so each panel is a bare plot
(axes, ticks, axis labels, data), and its legend is written next to it as a
standalone horizontal strip: `longitudinal_legend.png` and
`cross_context_legend.png`. Titles belong in
`output_data/connectome_figure.svg`, where they can be typeset once for the
whole multipanel figure; the coverage caveat for panel 3 is written as plain
text to `network_quality_note.txt` for the same reason. Place the legend
strips in Inkscape like any other panel — they are linked by relative path
and sized through `panel_size` exactly as the panels are.

Each is saved at exactly the size `output_data/connectome_figure.svg`
allocates it (`airoh.figures.panel_size`), at the montage's DPI, with
`layout="constrained"` and never `bbox_inches="tight"` — the two rules that
keep panel placement 1:1 (CLAUDE.md, "Figures: the Inkscape montage
pattern"). The detailed per-network histogram grids (from
`pair_histograms.tsv`) are saved alongside as extra diagnostic outputs in the
same folder, since that folder is this notebook's "already ran" sentinel.


In [1]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from airoh.figures import panel_size

FIGURE_DPI = int(os.environ.get("FIGURE_MONTAGE_DPI", 300))

output_dir = Path(os.environ.get("OUTPUT_DATA_DIR", "../output_data")).resolve()
figures_base = Path(os.environ.get("FIGURES_DIR", output_dir / "figures")).resolve()

project_root = output_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

figure_dir = figures_base / "figure_connectomes"
figure_dir.mkdir(parents=True, exist_ok=True)

with open(project_root / "invoke.yaml") as handle:
    invoke_config = yaml.safe_load(handle)

PARCELLATION = invoke_config["parcellation"]
NETWORK_ORDER = invoke_config["parcellations"][PARCELLATION]["network_order"]
MEASURE = invoke_config.get("analysis_measure", "pearson")

group_stats_dir = output_dir / "group_stats"
cross_context = pd.read_csv(group_stats_dir / "cross_context.tsv", sep="\t")
longitudinal_bins = pd.read_csv(group_stats_dir / "longitudinal_bins.tsv", sep="\t")
longitudinal_lag = pd.read_csv(group_stats_dir / "longitudinal_lag.tsv", sep="\t")
network_quality = pd.read_csv(group_stats_dir / "network_quality.tsv", sep="\t")
session_gate = pd.read_csv(group_stats_dir / "session_gate.tsv", sep="\t")
pair_histograms = pd.read_csv(group_stats_dir / "pair_histograms.tsv", sep="\t")
domain_cross_context = pd.read_csv(group_stats_dir / "domain_cross_context.tsv", sep="\t")

print(f"📂 {PARCELLATION}, measure={MEASURE}: "
      f"{len(cross_context)} cross-context rows, {len(longitudinal_bins)} longitudinal-bin rows")


📂 cneuromod2026, measure=pearson: 72 cross-context rows, 72 longitudinal-bin rows


In [2]:
# Legends live outside the panels, as their own montage elements.
#
# A panel-sized canvas has no room for a nine-network key on top of the data,
# so each panel is drawn bare and its legend is written to a separate
# `*_legend.png`: a horizontal strip holding nothing but the key. The strip is
# a montage element like any other, so it is sized through `panel_size` and
# saved at the montage DPI with the same two rules (`layout="constrained"`,
# never `bbox_inches="tight"`).


def save_legend(handles, labels, name, default_size, ncol=None, fontsize=7):
    """Render `handles`/`labels` alone into `{name}` as a horizontal strip."""
    if not handles:
        return
    figsize = panel_size(f"figure_connectomes/{name}", default_size)
    fig = plt.figure(figsize=figsize, layout="constrained")
    fig.legend(
        handles,
        labels,
        loc="center",
        ncol=ncol or min(len(handles), 5),
        fontsize=fontsize,
        frameon=False,
    )
    fig.savefig(figure_dir / name, dpi=FIGURE_DPI)
    plt.close(fig)
    print(f"✅ wrote {figure_dir / name} at {figsize} in")


In [3]:
# Panel 1 — longitudinal.png: within-subject similarity vs. friends season lag.
figsize = panel_size("figure_connectomes/longitudinal.png", (4.0, 4.5))
fig, ax = plt.subplots(figsize=figsize, layout="constrained")

lag = longitudinal_lag[
    (longitudinal_lag["gate"] == "gated") & (longitudinal_lag["lag_type"] == "season")
]
colors = plt.cm.tab10(np.linspace(0, 1, len(NETWORK_ORDER)))

if len(lag):
    for network, color in zip(NETWORK_ORDER, colors):
        within = lag[(lag["network"] == network) & (lag["pair_type"] == "within-subject")]
        within = within.sort_values("lag_value")
        if len(within):
            ax.plot(within["lag_value"], within["median"], marker="o", color=color,
                    label=network, linewidth=1.2, markersize=3)

    between = lag[(lag["pair_type"] == "between-subject")]
    if len(between):
        band = between.groupby("lag_value")["median"].median()
        ax.axhspan(band.min(), band.max(), color="0.85", zorder=0,
                    label="between-subject band")

    ax.set_xlabel("season lag")
    ax.set_ylabel("similarity (Fisher-z)")
else:
    ax.text(0.5, 0.5, "no friends longitudinal data\n(smoke run, or single season)",
            ha="center", va="center", transform=ax.transAxes, color="0.5", fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])

legend_handles, legend_labels = ax.get_legend_handles_labels()

fig.savefig(figure_dir / "longitudinal.png", dpi=FIGURE_DPI)
plt.close(fig)
print(f"✅ wrote {figure_dir / 'longitudinal.png'} at {figsize} in")

# Ten entries (nine networks plus the between-subject band): four columns
# over three rows keeps every label inside the strip.
save_legend(legend_handles, legend_labels, "longitudinal_legend.png", (6.0, 1.0), ncol=4)


✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/longitudinal.png at (2.283464566929134, 2.5590551181102366) in
✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/longitudinal_legend.png at (6.0, 1.0) in


In [4]:
# Panel 2 — cross_context.png: the four bins x nine networks, all datasets.
figsize = panel_size("figure_connectomes/cross_context.png", (5.0, 4.5))
fig, ax = plt.subplots(figsize=figsize, layout="constrained")

gated = cross_context[cross_context["gate"] == "gated"]
bin_order = [
    "within-subject / within-dataset", "within-subject / between-dataset",
    "between-subject / within-dataset", "between-subject / between-dataset",
]
bin_colors = {
    bin_order[0]: "#1b9e77", bin_order[1]: "#7570b3",
    bin_order[2]: "#d95f02", bin_order[3]: "#999999",
}

x = np.arange(len(NETWORK_ORDER))
width = 0.2
for i, bin_label in enumerate(bin_order):
    values = []
    for network in NETWORK_ORDER:
        row = gated[(gated["network"] == network) & (gated["bin"] == bin_label)]
        values.append(row["median"].iloc[0] if len(row) else np.nan)
    ax.bar(x + (i - 1.5) * width, values, width, color=bin_colors[bin_label],
           label=bin_label)

ax.set_xticks(x)
ax.set_xticklabels(NETWORK_ORDER, rotation=45, ha="right", fontsize=7)
ax.set_ylabel("median similarity (Fisher-z)")

legend_handles, legend_labels = ax.get_legend_handles_labels()

fig.savefig(figure_dir / "cross_context.png", dpi=FIGURE_DPI)
plt.close(fig)
print(f"✅ wrote {figure_dir / 'cross_context.png'} at {figsize} in")

# Four long labels: two columns keep the strip from running off the montage.
save_legend(legend_handles, legend_labels, "cross_context_legend.png", (5.0, 0.7), ncol=2)


✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/cross_context.png at (2.834645669291339, 2.5590551181102366) in
✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/cross_context_legend.png at (5.0, 0.7) in


In [5]:
# Panel 2b — domain_*.png: same four-bin comparison as cross_context.png,
# restricted to one naturalistic-stimulus domain at a time (CLAUDE.md,
# "Domain-restricted robustness figures") — movies (title-level task
# identity: friends season or movie10 title), videogames, stories
# (dataset-level task identity, same axis as cross_context.png).


def plot_domain_panel(domain, group_name, filename):
    """Draw one domain's four-bin x network grouped bar chart, save its legend."""
    figsize = panel_size(f"figure_connectomes/{filename}", (5.0, 4.5))
    fig, ax = plt.subplots(figsize=figsize, layout="constrained")

    subset = domain_cross_context[
        (domain_cross_context["domain"] == domain)
        & (domain_cross_context["gate"] == "gated")
    ]
    bin_order = [
        f"within-subject / within-{group_name}", f"within-subject / between-{group_name}",
        f"between-subject / within-{group_name}", f"between-subject / between-{group_name}",
    ]
    bin_colors = {
        bin_order[0]: "#1b9e77", bin_order[1]: "#7570b3",
        bin_order[2]: "#d95f02", bin_order[3]: "#999999",
    }

    x = np.arange(len(NETWORK_ORDER))
    width = 0.2
    for i, bin_label in enumerate(bin_order):
        values = []
        for network in NETWORK_ORDER:
            row = subset[(subset["network"] == network) & (subset["bin"] == bin_label)]
            values.append(row["median"].iloc[0] if len(row) else np.nan)
        ax.bar(x + (i - 1.5) * width, values, width, color=bin_colors[bin_label],
               label=bin_label)

    ax.set_xticks(x)
    ax.set_xticklabels(NETWORK_ORDER, rotation=45, ha="right", fontsize=7)
    ax.set_ylabel("median similarity (Fisher-z)")

    legend_handles, legend_labels = ax.get_legend_handles_labels()
    fig.savefig(figure_dir / filename, dpi=FIGURE_DPI)
    plt.close(fig)
    print(f"✅ wrote {figure_dir / filename} at {figsize} in")

    legend_name = filename.replace(".png", "_legend.png")
    save_legend(legend_handles, legend_labels, legend_name, (5.0, 0.7), ncol=2)


plot_domain_panel("movies", "title", "domain_movies.png")
plot_domain_panel("videogames", "dataset", "domain_videogames.png")
plot_domain_panel("stories", "dataset", "domain_stories.png")


✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/domain_movies.png at (5.0, 4.5) in
✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/domain_movies_legend.png at (5.0, 0.7) in


✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/domain_videogames.png at (5.0, 4.5) in
✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/domain_videogames_legend.png at (5.0, 0.7) in


✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/domain_stories.png at (5.0, 4.5) in
✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/domain_stories_legend.png at (5.0, 0.7) in


In [6]:
# Panel 3 — network_quality.png: within-subject stability vs. median tSNR per network.
# Single series, so no legend strip; the coverage caveat goes to a text file
# instead of onto the canvas, to be typeset as caption in the montage.
figsize = panel_size("figure_connectomes/network_quality.png", (3.5, 4.5))
fig, ax = plt.subplots(figsize=figsize, layout="constrained")

covered = network_quality[network_quality["n_tsnr"] > 0]
if len(covered) >= 2:
    ax.scatter(covered["median_tsnr"], covered["within_subject_median_cross_context"],
               color="#1b9e77", s=30)
    for _, row in covered.iterrows():
        point = (row["median_tsnr"], row["within_subject_median_cross_context"])
        ax.annotate(row["network"], point, fontsize=6,
                    xytext=(3, 3), textcoords="offset points")
    ax.set_xlabel("median tSNR")
    ax.set_ylabel("within-subject similarity")
    coverage_note = ""
else:
    order = network_quality.sort_values(
        "within_subject_median_cross_context", ascending=False
    )
    ax.barh(order["network"], order["within_subject_median_cross_context"], color="#1b9e77")
    ax.invert_yaxis()
    ax.set_xlabel("within-subject similarity")
    coverage_note = (
        f"tSNR coverage too thin ({int(covered.shape[0])}/{len(network_quality)} networks) "
        "— ordering only. Expected to fill in as qa_figures' atlas_tsnr tables land upstream."
    )

fig.savefig(figure_dir / "network_quality.png", dpi=FIGURE_DPI)
plt.close(fig)
print(f"✅ wrote {figure_dir / 'network_quality.png'} at {figsize} in")

note_path = figure_dir / "network_quality_note.txt"
note_path.write_text(coverage_note + "\n" if coverage_note else "")
print(f"✅ wrote {note_path}: {coverage_note or '(no caveat — tSNR coverage sufficient)'}")


✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/network_quality.png at (2.1653543307086616, 2.5590551181102366) in
✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/network_quality_note.txt: (no caveat — tSNR coverage sufficient)


In [7]:
# Diagnostic outputs (not montage panels): 3x3 density grids per analysis/measure,
# from the precomputed pair_histograms.tsv. Free-size, saved at 150 dpi.
for analysis, gate in (("cross_context", "gated"), ("longitudinal", "gated")):
    subset = pair_histograms[
        (pair_histograms["analysis"] == analysis) & (pair_histograms["gate"] == gate)
    ]
    if subset.empty:
        continue
    fig, axes = plt.subplots(3, 3, figsize=(12, 10), layout="constrained")
    fig.suptitle(f"{analysis} similarity distributions — {MEASURE}, gate={gate}")
    for network, sub_ax in zip(NETWORK_ORDER, axes.flat):
        net_hist = subset[subset["network"] == network]
        for bin_label, group in net_hist.groupby("bin"):
            group = group.sort_values("bin_left")
            centers = (group["bin_left"] + group["bin_right"]) / 2
            total = group["count"].sum()
            density = group["count"] / total if total else group["count"]
            sub_ax.plot(centers, density, label=bin_label, linewidth=1)
        sub_ax.set_title(network, fontsize=9)
        sub_ax.set_yticks([])
    axes.flat[0].legend(fontsize=6, loc="upper right")
    out_path = figure_dir / f"{analysis}_{MEASURE}_histograms.png"
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    print(f"✅ wrote {out_path}")


✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/cross_context_pearson_histograms.png


✅ wrote /home/pbellec/cneuromod.all.connectome_stats/output_data/figures/figure_connectomes/longitudinal_pearson_histograms.png
